# MDC vNext — Deep Baseline (Dense Autoencoder)

**Closes:** ISS-09 (`docs/ISSUES_AND_IMPROVEMENT_PLAN.md` §3.1) — "thin baselines: Isolation Forest only".
**Self-contained:** upload only this notebook. Loads `windows_vnext.npz` from `mdc_preprocess_vNext.ipynb`
and (optionally) `metrics_vnext.json` / `baseline_comparison.json` from the model / drift-aware runs to
build one combined comparison table: **Isolation Forest vs Dense AE vs vNext default vs vNext HPO**,
all evaluated on the identical test windows.

**Why Dense AE, not LSTM-AE:** the WBS's own fallback table (§8.0) lists Dense AE as the acceptable
minimum second baseline ("No time for Dense AE baseline → Isolation Forest only (minimum)"). A
feed-forward autoencoder has far fewer moving parts than an LSTM-AE (no hidden-state handling, no
sequence-length sensitivity), which matters here because this notebook is being handed off unexecuted
(no GPU in the authoring environment) — fewer moving parts means fewer places for an unverified bug
to hide before you run it on Colab.

**Protocol parity with vNext (for a fair comparison):**
- Same `windows_vnext.npz` (same train/val/test windows, same benign-only training set)
- Same auto invert-flip check on val (reconstruction error can come out anti-correlated with "attack")
- Same F1-optimal threshold rule (tuned on val, applied to test)
- Same metric set as the existing Isolation Forest baseline (ROC-AUC, PR-AUC, F1, MCC, precision, recall, FPR)

**KAGGLE version:** no Google Drive, no Colab. Loads `windows_vnext.npz` from attached input data (or a manual upload), writes outputs to `/kaggle/working/baselines/`, and zips + offers a browser download at the end -- see `mdc_preprocess_vNext_kaggle.ipynb`'s title cell for the handoff pattern between notebooks.


## 0. Environment

In [ ]:
import sys, os, subprocess
_IN_KAGGLE = os.path.exists('/kaggle/working')
if _IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'scikit-learn', 'matplotlib'], check=False)
print(f'In Kaggle: {_IN_KAGGLE}')


## 1. Configuration

In [ ]:
from __future__ import annotations
import os, json, gc, random, math
from pathlib import Path
import numpy as np

VERSION = 'vnext'
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Dense AE architecture (deliberately simple -- this is a baseline, not the main model)
HIDDEN_DIM      = 256
BOTTLENECK_DIM  = 64
DROPOUT         = 0.1
BATCH_SIZE      = 64
MAX_EPOCHS      = 60
LR              = 1e-3
WEIGHT_DECAY    = 1e-5
AUC_CHECK_FREQ  = 2
AUTO_SCORE_FLIP = True

NPZ_FILE = f'windows_{VERSION}.npz'

DATA_DIR     = Path('/kaggle/working/processed') if _IN_KAGGLE else Path('../outputs/vNEXT_test/processed')
RUN_DIR      = Path('/kaggle/working/runs')       if _IN_KAGGLE else Path('../outputs/vNEXT_test/runs')
BASELINE_DIR = Path('/kaggle/working/baselines')  if _IN_KAGGLE else Path('../outputs/vNEXT_test/baselines')

for _p in (DATA_DIR, RUN_DIR, BASELINE_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print('Config loaded')
print(f'  DATA_DIR (use)     : {DATA_DIR}')
print(f'  RUN_DIR  (use)     : {RUN_DIR}')
print(f'  BASELINE_DIR (use) : {BASELINE_DIR}')
print(f'  ARCH: hidden={HIDDEN_DIM} bottleneck={BOTTLENECK_DIM} dropout={DROPOUT}')


## 2. Mount drive + load npz + existing metrics

In [ ]:
# --- Kaggle IO helpers (self-contained -- no Drive, no Colab) ---
import os, sys, glob, shutil, zipfile, base64
from pathlib import Path

_IN_KAGGLE = os.path.exists('/kaggle/working')

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')

def kaggle_find(name, extra_dirs=()):
    """Search /kaggle/input/**, /kaggle/working/**, and extra_dirs for a file by
    name. There is no live shared Drive on Kaggle -- to hand a file from one
    notebook to the next, either (a) attach the producing notebook's own
    output via '+ Add Data > Your Notebooks' (no manual zip needed), or
    (b) download this notebook's output zip and upload it as a new Kaggle
    Dataset, then attach that dataset. Either way it shows up under
    /kaggle/input/<name>/ and this function finds it there."""
    roots = [KAGGLE_INPUT, KAGGLE_WORKING, *[Path(d) for d in extra_dirs]]
    for root in roots:
        if not root.is_dir():
            continue
        hits = sorted(glob.glob(str(root / '**' / name), recursive=True))
        if hits:
            return Path(hits[0])
    return None

def kaggle_upload_fallback(name, dest_dir):
    """Best-effort interactive upload widget for a live session. Not available
    during a headless 'Save & Run All' commit (no UI) -- attach the file as
    input data instead in that case."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
        uploader = widgets.FileUpload(accept='', multiple=False)
        display(uploader)
        print(f'Upload {name} with the widget above, then re-run this cell.')
        if uploader.value:
            item = list(uploader.value.values())[0]
            content = item['content'] if isinstance(item, dict) else item.content
            dest = Path(dest_dir) / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(bytes(content))
            print(f'Saved -> {dest}')
            return dest
    except Exception as e:
        print(f'Interactive upload unavailable ({e}).')
    return None

def zip_and_offer_download(src_dir, zip_name, max_auto_mb=25):
    """Zip src_dir into /kaggle/working/{zip_name}.zip and try to trigger a
    browser download automatically. Only fires in a live, actively-open
    browser tab (not during headless 'Save & Run All') -- Kaggle's own Output
    tab (right sidebar) always lists this zip for manual download regardless
    of whether the auto-download trick actually fires in your browser."""
    src_dir = Path(src_dir)
    zip_base = KAGGLE_WORKING / zip_name
    zip_path = zip_base.with_suffix('.zip')
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_base), 'zip', root_dir=src_dir)
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f'Zipped -> {zip_path}  ({size_mb:.1f} MB)')
    if size_mb <= max_auto_mb:
        try:
            from IPython.display import HTML, display
            b64 = base64.b64encode(zip_path.read_bytes()).decode()
            html = (
                f'<a id="dl_{zip_name}" download="{zip_path.name}" '
                f'href="data:application/zip;base64,{b64}"></a>'
                f'<script>document.getElementById("dl_{zip_name}").click();</script>'
            )
            display(HTML(html))
            print('Auto-download triggered (only works if this tab is actively open --')
            print('if nothing happened, use the Output tab on the right instead).')
        except Exception as e:
            print(f'Auto-download trick failed ({e}) -- use the Output tab instead.')
    else:
        print(f'{size_mb:.1f} MB exceeds the {max_auto_mb} MB auto-download guard -- '
              'skipping the browser trick to avoid bloating notebook output.')
        print('Get it from the Output tab (right sidebar) after Save Version instead.')
    return zip_path

print(f'Kaggle IO ready. In Kaggle: {_IN_KAGGLE}')

npz_path = kaggle_find(NPZ_FILE)
if npz_path is None:
    print(f'{NPZ_FILE} not found under /kaggle/input or /kaggle/working.')
    print("Attach mdc_preprocess_vNext_kaggle.ipynb's output via '+ Add Data', or upload it now:")
    npz_path = kaggle_upload_fallback(NPZ_FILE, DATA_DIR)

if npz_path is None:
    raise FileNotFoundError(
        f'{NPZ_FILE} not found under /kaggle/input or /kaggle/working, and no valid '
        "file was uploaded. Run mdc_preprocess_vNext_kaggle.ipynb first and attach its "
        "output via '+ Add Data', or upload the npz manually."
    )

z = np.load(npz_path, allow_pickle=True)
X_train = z['X_train'].astype(np.float32)
X_val   = z['X_val'].astype(np.float32)
X_test  = z['X_test'].astype(np.float32)
y_val   = z['y_val'].astype(np.int8)
y_test  = z['y_test'].astype(np.int8)
T, n_features = X_test.shape[1], X_test.shape[2]
input_dim = T * n_features
z.close()
print(f'Loaded {npz_path.name}: train={X_train.shape} val={X_val.shape} test={X_test.shape}  input_dim={input_dim}')

# Optional: pull existing vNext metrics + IF baseline for the combined table (not required to train)
metrics_path = kaggle_find('metrics_vnext.json')
metrics_vnext = json.loads(metrics_path.read_text()) if metrics_path else None
print(f'metrics_vnext.json: {"found" if metrics_vnext else "NOT FOUND (vNext rows will be blank in the table)"}')

if_baseline_path = kaggle_find('baseline_comparison.json')
if_baseline = json.loads(if_baseline_path.read_text()) if if_baseline_path else None
print(f'baseline_comparison.json (IF): {"found" if if_baseline else "NOT FOUND (run mdc_drift_aware_kaggle.ipynb first for the IF row)"}')


## 3. Tensors / device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_flat).float()),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0,
)
print(f'Train batches: {len(train_loader)}  val windows: {len(X_val_flat)}  test windows: {len(X_test_flat)}')


## 4. Dense autoencoder

Flatten each `(T, F)` window to a single `T*F` vector and reconstruct it through a
small bottleneck MLP. Reconstruction MSE is the anomaly score (higher = more anomalous),
same convention as the main vNext transformer AE.

In [ ]:
class DenseAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, bottleneck_dim=BOTTLENECK_DIM, dropout=DROPOUT):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, bottleneck_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

def build_dense_ae():
    return DenseAE(input_dim).to(device)

print('DenseAE defined:', f'{input_dim} -> {HIDDEN_DIM} -> {BOTTLENECK_DIM} -> {HIDDEN_DIM} -> {input_dim}')


## 5. Scoring helpers

Same auto invert-flip convention as the main pipeline: reconstruction error can come out
anti-correlated with "attack" depending on what the model learns, so both directions are
checked on val and the better one is used (never assumed).

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, confusion_matrix,
)

@torch.no_grad()
def compute_scores_dense(m, X_flat, bs=256):
    m.eval()
    out = []
    for k in range(0, len(X_flat), bs):
        xb = torch.from_numpy(X_flat[k:k+bs]).float().to(device)
        rb = m(xb)
        err = (rb - xb).pow(2).mean(dim=1)
        out.append(err.cpu().numpy())
        del xb, rb, err
    return np.concatenate(out)

def thresholds_from_val_dense(scores, y):
    order = np.argsort(scores)
    s_sorted = scores[order]
    best_f1, best_thr = -1.0, float(s_sorted[0])
    for thr in np.unique(s_sorted):
        pred = (scores >= thr).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr

def full_eval_dense(scores, y, thr):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'roc_auc': float(roc_auc_score(y, scores)),
        'pr_auc': float(average_precision_score(y, scores)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
        'precision': float(tp / max(tp + fp, 1)),
        'recall': float(tp / max(tp + fn, 1)),
        'fpr': float(fp / max(fp + tn, 1)),
        'threshold': float(thr),
    }

print('scoring helpers ready')


## 6. Train (benign-only, same convention as vNext)

In [ ]:
model = build_dense_ae()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_auc = -1.0
best_state = None
history = []

y_val_t = y_val  # already int8 array

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(device)
        opt.zero_grad(set_to_none=True)
        rb = model(xb)
        loss = F.mse_loss(rb, xb)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(X_train_flat)

    if epoch % AUC_CHECK_FREQ == 0 or epoch == MAX_EPOCHS:
        s_val_raw = compute_scores_dense(model, X_val_flat)
        auc_raw  = roc_auc_score(y_val_t, s_val_raw)
        auc_flip = roc_auc_score(y_val_t, -s_val_raw)
        auc = max(auc_raw, auc_flip)
        history.append({'epoch': epoch, 'loss': epoch_loss, 'val_auc': auc})
        if auc > best_val_auc:
            best_val_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch:>3}  loss={epoch_loss:.5f}  val_auc={auc:.4f}  best={best_val_auc:.4f}')

if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nTraining done. best_val_auc={best_val_auc:.4f}')


## 7. Evaluate on test (F1-optimal threshold, same protocol as IF baseline)

In [ ]:
s_val_raw  = compute_scores_dense(model, X_val_flat)
s_test_raw = compute_scores_dense(model, X_test_flat)

val_raw  = roc_auc_score(y_val,  s_val_raw)
val_flip = roc_auc_score(y_val, -s_val_raw)
invert = bool(AUTO_SCORE_FLIP and val_flip > val_raw)
print(f'val AUC raw={val_raw:.4f} flip={val_flip:.4f} -> invert={invert}')

s_val  = -s_val_raw  if invert else s_val_raw
s_test = -s_test_raw if invert else s_test_raw

thr = thresholds_from_val_dense(s_val, y_val)
dense_ae_result = full_eval_dense(s_test, y_test, thr)
dense_ae_result['score_inverted'] = invert
dense_ae_result['best_val_auc'] = float(best_val_auc)

print('\nDense AE baseline -- test metrics (F1-optimal threshold):')
for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr'):
    print(f'  {k:<10} {dense_ae_result[k]:.4f}')


## 8. Combined comparison table (ISS-09)

`Isolation Forest` (from `mdc_drift_aware.ipynb`) vs `Dense AE` (this notebook) vs
`vNext default` / `vNext HPO` (from `mdc_model_vNext.ipynb`) -- all on the identical
test windows from `windows_vnext.npz`, so the comparison is apples-to-apples.

In [ ]:
rows = [{'config': 'Dense AE (this notebook)', **{k: dense_ae_result[k] for k in
         ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')}, 'source': 'mdc_baselines.ipynb'}]

if if_baseline is not None:
    r_if = if_baseline.get('isolation_forest_mean_max', {}).get('metrics', {})
    if r_if:
        rows.append({'config': 'Isolation Forest (mean_max pool)',
                      **{k: r_if.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_drift_aware.ipynb'})

if metrics_vnext is not None:
    r_def = metrics_vnext.get('test_default', {}).get('f1_optimal', {})
    if r_def:
        rows.append({'config': 'vNext default (Transformer AE)',
                      **{k: r_def.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_model_vNext.ipynb'})
    r_hpo = metrics_vnext.get('test_hpo', {})
    if r_hpo:
        rows.append({'config': 'vNext HPO best (Transformer AE)',
                      **{k: r_hpo.get(k) for k in ('roc_auc', 'pr_auc', 'f1', 'mcc', 'precision', 'recall', 'fpr')},
                      'source': 'mdc_model_vNext.ipynb'})

import pandas as pd
comparison_df = pd.DataFrame(rows)
print('\n=== Deep + classical baseline comparison (ISS-09) ===')
print(comparison_df.to_string(index=False))

if len(rows) < 4:
    _missing = {'Isolation Forest'} if if_baseline is None else set()
    if metrics_vnext is None:
        _missing |= {'vNext default', 'vNext HPO'}
    print(f'\nNOTE: table is partial -- run these notebooks first to fill it in: {sorted(_missing)}'
          if _missing else '')

out = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'dense_ae': dense_ae_result,
    'comparison_table': comparison_df.to_dict(orient='records'),
    'protocol': {
        'same_windows_vnext_npz': True,
        'threshold_rule': 'f1_optimal_on_val',
        'auto_score_flip': AUTO_SCORE_FLIP,
    },
}
out_json = BASELINE_DIR / 'deep_baseline_comparison.json'
out_csv  = BASELINE_DIR / 'deep_baseline_comparison.csv'
out_json.write_text(json.dumps(out, indent=2))
comparison_df.to_csv(out_csv, index=False)
print(f'\nSaved -> {out_json}')
print(f'Saved -> {out_csv}')


## 9. Zip results for download (Kaggle -- no Drive)


In [ ]:
zip_and_offer_download(BASELINE_DIR, 'deep_baseline_comparison')
